In [0]:
# Bronze to Silver
# 1. Remove Duplicates
# 2. Validate Data 
# 3. Clean Data
# 4. Standardize Schemas
# 5. Enterprise Joins
from pyspark.sql import SparkSession

# Variables
catalog = "new_databricks_workspace"
schema = "default"
# Create SparkSession
spark = SparkSession.builder.appName("Spark Dataframe").getOrCreate()

# Read in both the csv's and create DataFrames
# InferSchema = False, I want to do it myself
df1 = spark.read.csv("/Volumes/new_databricks_workspace/default/athlete_volume/athlete_information.csv", sep=',', header=True)
# InferSchema = True, I want Spark to infer the data types
df2 = spark.read.csv("/Volumes/new_databricks_workspace/default/athlete_volume/athlete_results.csv", sep=',', header=True, inferSchema=True)

# Lets cast the columns to the correct data types for DataFrame 1
df1 = df1.withColumn("athlete_id", df1.athlete_id.cast("int"))
df1 = df1.withColumn("age", df1.age.cast("float"))
df1 = df1.withColumn("height", df1.height.cast("float"))
df1 = df1.withColumn("arm_span", df1.arm_span.cast("float"))
df1 = df1.withColumn("birthday", df1.birthday.cast("date"))

# Lets join both dataframes and create one large DataFrame
combined_df = df1.join(df2, df1.athlete_id == df2.athlete_id, "fullouter")

# Write the DataFrames to the Silver tables
df1.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_athlete_information")
df2.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_athlete_results")
# An extra step to drop the duplicate athlete_id column
combined_clean = combined_df.drop(df2["athlete_id"])
combined_clean.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_athlete_combined")

print("Silver tables created successfully")